# 03 — Anomaly Detection

Apply Isolation Forest to flag unusual intraday liquidity transactions.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_processing import load_raw_data, preprocess
from src.feature_engineering import build_features, get_feature_matrix, NUMERIC_FEATURES
from src.anomaly_detection import AnomalyDetector

%matplotlib inline
sns.set_theme(style='whitegrid')


## 1. Load and prepare features

In [ ]:
df_raw = load_raw_data('../data/raw/synthetic_intraday_payments_500k.csv')
df = preprocess(df_raw)

# Use a representative sample for speed
df_sample = df.sample(50_000, random_state=42).copy()
df_features = build_features(df_sample)
X, feature_names = get_feature_matrix(df_features)
print(f'Feature matrix shape: {X.shape}')
print(f'Features: {feature_names}')


## 2. Fit Isolation Forest

In [ ]:
detector = AnomalyDetector(method='isolation_forest', contamination=0.02)
detector.fit(X)
print('Detector fitted.')


## 3. Label anomalies

In [ ]:
df_labelled = detector.label_dataframe(df_features, feature_names)
n_anomalies = (df_labelled['anomaly_label'] == -1).sum()
print(f'Anomalies detected: {n_anomalies} ({n_anomalies / len(df_labelled):.1%})')
df_labelled[['transaction_id', 'timestamp', 'bank_id', 'amount', 'anomaly_label', 'anomaly_score']].head(10)


## 4. Visualise anomalies

In [ ]:
plot_sample = df_labelled.sample(5000, random_state=0)

fig, ax = plt.subplots(figsize=(10, 6))
normal = plot_sample[plot_sample['anomaly_label'] == 1]
anom   = plot_sample[plot_sample['anomaly_label'] == -1]

ax.scatter(normal['minute_of_day'], normal['amount'],
           c='steelblue', alpha=0.3, s=8, label='Normal')
ax.scatter(anom['minute_of_day'],   anom['amount'],
           c='red', alpha=0.6, s=12, label='Anomaly')

ax.set_xlabel('Minute of Day')
ax.set_ylabel('Transaction Amount (USD)')
ax.set_title('Anomaly Detection – Isolation Forest')
ax.legend()
plt.tight_layout()
plt.savefig('../figures/anomaly_detection.png', dpi=150)
plt.show()


## 5. Anomaly summary by bank and date

In [ ]:
summary = detector.anomaly_summary(df_labelled)
print(summary.head(10))


## 6. Save detector

In [ ]:
detector.save('../models/anomaly_detector.pkl')
print('Anomaly detector saved.')
